# Sales Analytics — Python Data Cleaning, Transformation & EDA

**Dataset:** Superstore Data 1.csv  
**Records:** 9,994  
**Columns:** 25

### Objective
Use Python, Pandas and NumPy for data cleaning, transformation, validation and exploratory data analysis. SQL is used separately for structured business analysis, while Power BI is used for the final interactive dashboard.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme()


## 2. Load Dataset

In [ ]:
# Local file used during project development
file_path = r"C:\Users\ASUS\Downloads\Superstore Data 1.csv"

df = pd.read_csv(file_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()


## 3. Initial Data Inspection

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nFirst 5 Records:")
display(df.head())


## 4. Data Quality Checks

In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0])

# Complete duplicate rows
print("Complete duplicate rows:", df.duplicated().sum())

# Check key categorical fields
categorical_cols = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].dropna().unique())
    print()


## 5. Data Cleaning & Transformation

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip()

# Convert dates
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")

# Standardize text fields
text_cols = [
    "Ship Mode", "Segment", "Country", "City", "State",
    "Region", "Category", "Sub-Category", "Product Name"
]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Ensure numerical fields have appropriate data types
numeric_cols = ["Sales", "Quantity", "Discount", "Profit", "Postal Code"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Derived fields for analysis
df["Profit Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0
)

df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Time-based fields
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Month Name"] = df["Order Date"].dt.month_name()

print("Data transformation completed.")
display(df.head())


## 6. Validation After Cleaning

In [ ]:
print("Final Shape:", df.shape)
print("Missing values after transformation:", df.isna().sum().sum())
print("Complete duplicate rows:", df.duplicated().sum())

validation = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()
})

display(validation)


## 7. Descriptive Statistics

In [ ]:
display(df[["Sales", "Quantity", "Discount", "Profit", "Profit Margin", "Shipping Days"]].describe().T)


## 8. Sales Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="Sales", bins=40, kde=True)
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 9. Profit Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="Profit", bins=40, kde=True)
plt.title("Profit Distribution")
plt.xlabel("Profit")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 10. Sales vs Profit

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Sales",
    y="Profit",
    hue="Category",
    alpha=0.6
)
plt.title("Sales vs Profit")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()


## 11. Sales by Region

In [ ]:
region_analysis = (
    df.groupby("Region", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
      .sort_values("Sales", ascending=False)
)

display(region_analysis)

plt.figure(figsize=(9, 5))
sns.barplot(data=region_analysis, x="Region", y="Sales")
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## 12. Sales & Profit by Category

In [ ]:
category_analysis = (
    df.groupby("Category", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
      .sort_values("Sales", ascending=False)
)

display(category_analysis)

plt.figure(figsize=(9, 5))
sns.barplot(data=category_analysis, x="Category", y="Sales")
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## 13. Customer Segment Analysis

In [ ]:
segment_analysis = (
    df.groupby("Segment", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
      .sort_values("Sales", ascending=False)
)

display(segment_analysis)

plt.figure(figsize=(8, 5))
sns.barplot(data=segment_analysis, x="Segment", y="Sales")
plt.title("Sales by Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## 14. Monthly Sales Exploration

In [ ]:
monthly_sales = (
    df.groupby(["Order Year", "Order Month", "Order Month Name"], as_index=False)
      .agg(Sales=("Sales", "sum"))
      .sort_values(["Order Year", "Order Month"])
)

display(monthly_sales.head(12))

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=monthly_sales,
    x="Order Month",
    y="Sales",
    hue="Order Year",
    marker="o"
)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## 15. Sub-Category Exploration

In [ ]:
subcategory = (
    df.groupby("Sub-Category", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
      .sort_values("Sales", ascending=False)
)

display(subcategory)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=subcategory,
    x="Sales",
    y="Sub-Category"
)
plt.title("Sales by Sub-Category")
plt.xlabel("Sales")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.show()


## 16. Exploratory Findings

In [ ]:
highest_sales_region = region_analysis.iloc[0]
highest_sales_category = category_analysis.iloc[0]
highest_sales_segment = segment_analysis.iloc[0]
highest_sales_subcategory = subcategory.iloc[0]

print(f"Highest-sales region: {highest_sales_region['Region']} "
      f"({highest_sales_region['Sales']:,.2f})")

print(f"Highest-sales category: {highest_sales_category['Category']} "
      f"({highest_sales_category['Sales']:,.2f})")

print(f"Highest-sales segment: {highest_sales_segment['Segment']} "
      f"({highest_sales_segment['Sales']:,.2f})")

print(f"Highest-sales sub-category: {highest_sales_subcategory['Sub-Category']} "
      f"({highest_sales_subcategory['Sales']:,.2f})")


## Conclusion

Python was used primarily for **data cleaning, transformation, validation, and exploratory data analysis**.

The structured business questions and ranking analysis are handled separately in the SQL workflow, while the final insights are communicated through the Power BI dashboard.
